## TUNI 2024-I 

In [50]:
# Paquetes
from pathlib import Path

import unicodedata
import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns

In [51]:
# Rutas de Censo 1996
ruta_tuni = Path(r"E:\BD\SUNEDU\TUNI\corte_16092026\3_matriculados")

# Rutas de programas
ruta_correspondencias = Path(
    r"D:\Daniel_Pajita\1_Proyectos\Research\Educacion"
    r"\educacion-universitaria-peru\2_Datos"
    r"\Correspondencias_entre_clasificaciones"
)

#### Base de pregrado 2024

In [52]:
# Cargar todas las variables: Matrícula 2024-I
archivo_tuni_2024_I = ruta_tuni / "matriculado_2024_I.txt"
tuni_2024_I = pd.read_csv(archivo_tuni_2024_I,sep="|",encoding="cp1252",dtype="string",na_values=["<NULL>"],)

In [53]:
# Filtrar pregrado sin modificar la base completa
nivel_incluir = ["Carrera Profesional"]
tipos_entidad_incluir = ["Universidad", "Escuela de Posgrado"]

tuni_pregrado_2024_I = tuni_2024_I.loc[
    tuni_2024_I["TIPO_ENTIDAD"].str.strip().isin(tipos_entidad_incluir) & tuni_2024_I["NIVEL_ACADEMICO"].str.strip().isin(nivel_incluir)
    ].copy()

In [54]:
# Seleccionar las variables de análisis: pregrado TUNI 2024-I
variables_pregrado_2024 = {
    "GUID_PERSONA": "id_persona_original",
    "CODIGO_INEI": "codigo_universidad",
    "NOMBRE_ENTIDAD": "nombre_universidad",
    "TIPO_ENTIDAD": "tipo_entidad",
    "TIPO_GESTION": "tipo_universidad",
    "TIPO_CONSTITUCION": "tipo_constitucion",
    "LICENCIA": "licencia",
    "PERIODO": "periodo",
    "PERIODO_ESTANDARIZADO": "periodo_estandarizado",
    "NIVEL_ACADEMICO": "nivel_academico_original",
    "PERIODO_LECTIVO": "periodo_lectivo",
    "CODIGO_SIU_PROGRAMA": "codigo_programa",
    "NOMBRE_PROGRAMA": "nombre_programa",
    "CODIGO_GRUPO_1": "codigo_grupo_1",
    "NOMBRE_GRUPO_1": "nombre_grupo_1",
    "CODIGO_GRUPO_3": "codigo_grupo_3",
    "NOMBRE_GRUPO_3": "nombre_grupo_3",
    "SEXO": "sexo",
    "ANIO_NACIMIENTO": "anio_nacimiento_original",
    "EDAD": "edad",
    "NACIONALIDAD": "nacionalidad",
    "ES_LOCAL_PRINCIPAL": "es_local_principal",
    "CODIGO_LOCAL": "codigo_local",
    "DEPARTAMENTO_LOCAL": "departamento_estudios",
    "PROVINCIA_LOCAL": "provincia_estudios",
    "DISTRITO_LOCAL": "distrito_estudios",
    "CODIGO_UBIGEO_INEI_LOCAL": "codigo_territorial_estudios",
    "DEPARTAMENTO_NACIMIENTO": "departamento_nacimiento",
    "ANIO_PERIODO_INGRESO": "anio_periodo_ingreso",
    "FECHA_INICIO_PERIODO": "fecha_inicio_periodo",
    "FECHA_FIN_PERIODO": "fecha_fin_periodo",
}

pregrado_2024_analisis = (tuni_pregrado_2024_I[list(variables_pregrado_2024)].rename(columns=variables_pregrado_2024).copy())
pregrado_2024_analisis.insert(0, "id_fila", range(1, len(pregrado_2024_analisis) + 1))
pregrado_2024_analisis["nivel_academico"] = "Pregrado"

# Limpiar espacios en los campos de texto
for columna in pregrado_2024_analisis.select_dtypes(include=["object", "string"]).columns:
    pregrado_2024_analisis[columna] = (pregrado_2024_analisis[columna].astype("string").str.strip().replace("", pd.NA))

# Convertir la edad y el año de nacimiento a números
for columna in ["edad", "anio_nacimiento_original"]:
    pregrado_2024_analisis[columna] = pd.to_numeric(pregrado_2024_analisis[columna], errors="coerce").astype("Int64")

# Convertir las fechas del periodo
for columna in ["fecha_inicio_periodo", "fecha_fin_periodo"]:
    pregrado_2024_analisis[columna] = pd.to_datetime(pregrado_2024_analisis[columna],errors="coerce",)


In [55]:
pregrado_2024_analisis.head()

,id_fila,id_persona_original,codigo_universidad,nombre_universidad,tipo_entidad,tipo_universidad,tipo_constitucion,licencia,periodo,periodo_estandarizado,...,codigo_local,departamento_estudios,provincia_estudios,distrito_estudios,codigo_territorial_estudios,departamento_nacimiento,anio_periodo_ingreso,fecha_inicio_periodo,fecha_fin_periodo,nivel_academico
0,1,{3B60CFFE-2AC6-4A8F-AB56-5C3F083E257B},260000061,Universidad Privada Norbert Wiener S.A.,Universidad,Privado,Societaria- Con fines de lucro,Licenciada,2024-1,2024-1,...,SL01,Lima,Lima,Lima,150101,Lima,2022,2024-03-01,2024-08-15,Pregrado
1,2,{9A5E8784-FB99-48EA-9C7A-78070660718B},260000061,Universidad Privada Norbert Wiener S.A.,Universidad,Privado,Societaria- Con fines de lucro,Licenciada,2024-1,2024-1,...,SL01,Lima,Lima,Lima,150101,Lima,2023,2024-03-01,2024-08-15,Pregrado
2,3,{E1AE694D-0E02-448F-81CF-34DAAA3EE04E},160000002,Universidad Nacional de San Cristóbal de Huamanga,Universidad,Público,Pública,Licenciada,2024-1,2024-1,...,SL01,Ayacucho,Huamanga,Ayacucho,50101,Ayacucho,2006,2023-09-19,2024-04-03,Pregrado
3,4,{3957D822-CC90-414F-8C08-96E75BBE76E9},160000002,Universidad Nacional de San Cristóbal de Huamanga,Universidad,Público,Pública,Licenciada,2024-1,2024-1,...,SL01,Ayacucho,Huamanga,Ayacucho,50101,Apurímac,2019,2023-09-19,2024-04-03,Pregrado
4,5,{A4D8C870-27BC-4A03-BB7E-E708AE5A01B8},160000002,Universidad Nacional de San Cristóbal de Huamanga,Universidad,Público,Pública,Licenciada,2024-1,2024-1,...,SL01,Ayacucho,Huamanga,Ayacucho,50101,Ayacucho,2022,2023-09-19,2024-04-03,Pregrado


In [56]:
# Edad promedio por sexo y tipo de universidad: pregrado TUNI 2024-I
tabla_edad_pregrado = pd.pivot_table(
    pregrado_2024_analisis,
    values="edad",
    index="sexo",
    columns="tipo_universidad",
    aggfunc="mean",
    margins=True,
    margins_name="TOTAL",
)

display(tabla_edad_pregrado.style.format("{:.2f}", na_rep="—"))

tipo_universidad,Privado,Público,TOTAL
sexo,,,
Femenino,22.94,21.68,22.68
Masculino,23.77,22.51,23.44
TOTAL,23.33,22.14,23.05


#### Base de posgrado 2024

In [57]:
# Filtrar posgrado sin modificar la base completa
niveles_posgrado = ["Segunda Especialidad", "Maestría", "Doctorado"]
tipos_entidad_incluir = ["Universidad", "Escuela de Posgrado"]

tuni_posgrado_2024_I = tuni_2024_I.loc[
    tuni_2024_I["TIPO_ENTIDAD"].str.strip().isin(tipos_entidad_incluir) & tuni_2024_I["NIVEL_ACADEMICO"].str.strip().isin(niveles_posgrado)
].copy()

In [58]:
# Seleccionar las variables de análisis: posgrado TUNI 2024-I
variables_posgrado_2024 = {
    "GUID_PERSONA": "id_persona_original",
    "CODIGO_INEI": "codigo_universidad",
    "NOMBRE_ENTIDAD": "nombre_universidad",
    "TIPO_ENTIDAD": "tipo_entidad",
    "TIPO_GESTION": "tipo_universidad",
    "TIPO_CONSTITUCION": "tipo_constitucion",
    "LICENCIA": "licencia",
    "PERIODO": "periodo",
    "PERIODO_ESTANDARIZADO": "periodo_estandarizado",
    "NIVEL_ACADEMICO": "nivel_academico_original",
    "PERIODO_LECTIVO": "periodo_lectivo",
    "CODIGO_SIU_PROGRAMA": "codigo_programa",
    "NOMBRE_PROGRAMA": "nombre_programa",
    "CODIGO_GRUPO_1": "codigo_grupo_1",
    "NOMBRE_GRUPO_1": "nombre_grupo_1",
    "CODIGO_GRUPO_3": "codigo_grupo_3",
    "NOMBRE_GRUPO_3": "nombre_grupo_3",
    "SEXO": "sexo",
    "ANIO_NACIMIENTO": "anio_nacimiento_original",
    "EDAD": "edad",
    "NACIONALIDAD": "nacionalidad",
    "ES_LOCAL_PRINCIPAL": "es_local_principal",
    "CODIGO_LOCAL": "codigo_local",
    "DEPARTAMENTO_LOCAL": "departamento_estudios",
    "PROVINCIA_LOCAL": "provincia_estudios",
    "DISTRITO_LOCAL": "distrito_estudios",
    "CODIGO_UBIGEO_INEI_LOCAL": "codigo_territorial_estudios",
    "DEPARTAMENTO_NACIMIENTO": "departamento_nacimiento",
    "ANIO_PERIODO_INGRESO": "anio_periodo_ingreso",
    "FECHA_INICIO_PERIODO": "fecha_inicio_periodo",
    "FECHA_FIN_PERIODO": "fecha_fin_periodo",
}

posgrado_2024_analisis = (tuni_posgrado_2024_I[list(variables_posgrado_2024)].rename(columns=variables_posgrado_2024).copy())
posgrado_2024_analisis.insert(0, "id_fila", range(1, len(posgrado_2024_analisis) + 1))

# Limpiar espacios en los campos de texto
for columna in posgrado_2024_analisis.select_dtypes(include=["object", "string"]).columns:
    posgrado_2024_analisis[columna] = (posgrado_2024_analisis[columna].astype("string").str.strip().replace("", pd.NA))

# Identificar el nivel académico desagregado
posgrado_2024_analisis["nivel_academico"] = (
    posgrado_2024_analisis["nivel_academico_original"].map({
        "Segunda Especialidad": "Segunda especialidad",
        "Maestría": "Maestría",
        "Doctorado": "Doctorado",
    }).fillna("No especifica")
)

# Convertir la edad y el año de nacimiento a números
for columna in ["edad", "anio_nacimiento_original"]:
    posgrado_2024_analisis[columna] = pd.to_numeric(posgrado_2024_analisis[columna],errors="coerce",).astype("Int64")

# Convertir las fechas del periodo
for columna in ["fecha_inicio_periodo", "fecha_fin_periodo"]:
    posgrado_2024_analisis[columna] = pd.to_datetime(posgrado_2024_analisis[columna],errors="coerce",)


In [59]:
posgrado_2024_analisis.head()

,id_fila,id_persona_original,codigo_universidad,nombre_universidad,tipo_entidad,tipo_universidad,tipo_constitucion,licencia,periodo,periodo_estandarizado,...,codigo_local,departamento_estudios,provincia_estudios,distrito_estudios,codigo_territorial_estudios,departamento_nacimiento,anio_periodo_ingreso,fecha_inicio_periodo,fecha_fin_periodo,nivel_academico
12153,1,{DC8FCFE7-2FDE-4BEE-92BA-F6653078007F},260000017,Universidad del Pacífico,Universidad,Privado,Asociativa- Sin fines de lucro,Licenciada,2024-1,2024-1,...,SL01,Lima,Lima,Jesús María,150113,Lima,2023,2023-11-01,2026-02-28,Maestría
12154,2,{06F69C3D-A588-409A-99E7-C4BEAF615F5D},260000065,Universidad Tecnológica del Perú S.A.C.,Universidad,Privado,Societaria- Con fines de lucro,Licenciada,2024-1,2024-1,...,SL07,Lima,Lima,San Isidro,150131,Lambayeque,2023,2024-01-02,2024-07-31,Maestría
12155,3,{03DDDA81-FE90-4FEE-A2D1-C31570F88B5C},260000065,Universidad Tecnológica del Perú S.A.C.,Universidad,Privado,Societaria- Con fines de lucro,Licenciada,2024-1,2024-1,...,SL07,Lima,Lima,San Isidro,150131,Piura,2023,2024-01-02,2024-07-31,Maestría
12156,4,{558A34B3-C340-43DF-96DC-52F78AA60C37},260000065,Universidad Tecnológica del Perú S.A.C.,Universidad,Privado,Societaria- Con fines de lucro,Licenciada,2024-1,2024-1,...,SL07,Lima,Lima,San Isidro,150131,Piura,2023,2024-01-02,2024-07-31,Maestría
12157,5,{6CC312A7-37DE-4102-BFDB-31F506065C35},260000065,Universidad Tecnológica del Perú S.A.C.,Universidad,Privado,Societaria- Con fines de lucro,Licenciada,2024-1,2024-1,...,SL07,Lima,Lima,San Isidro,150131,Lima,2023,2024-01-02,2024-07-31,Maestría


In [60]:
# Edad promedio por sexo, tipo de universidad y nivel académico: posgrado 2024-I
tabla_edad_posgrado = pd.pivot_table(
    posgrado_2024_analisis,
    values="edad",
    index=["nivel_academico", "sexo"],
    columns="tipo_universidad",
    aggfunc="mean",
    margins=True,
    margins_name="TOTAL",
)

display(tabla_edad_posgrado.style.format("{:.2f}", na_rep="—"))

In [61]:
# Tabla resumen de pregrado y posgrado: TUNI 2024-I
datos_tabla = pd.concat(
    [
        pregrado_2024_analisis[["nivel_academico", "sexo", "tipo_universidad", "edad"]],
        posgrado_2024_analisis[["nivel_academico", "sexo", "tipo_universidad", "edad"]],
    ],
    ignore_index=True,
)

datos_tabla["Sexo"] = (
    datos_tabla["sexo"].str.strip().str.casefold().map({
        "masculino": "Masculino",
        "femenino": "Femenino",
        "hombre": "Masculino",
        "mujer": "Femenino",
    })
)

datos_tabla["Tipo_universidad"] = (
    datos_tabla["tipo_universidad"].str.strip().str.casefold().map({
        "público": "Pública",
        "pública": "Pública",
        "privado": "Privada",
        "privada": "Privada",
    })
)

niveles = ["Pregrado", "Segunda especialidad", "Maestría", "Doctorado"]
datos_tabla = datos_tabla.loc[datos_tabla["nivel_academico"].isin(niveles)]

# Resumir cada combinación, incluyendo los totales
resultados = {}

for sexo in ["Masculino", "Femenino", "TOTAL"]:
    for tipo in ["Pública", "Privada", "TOTAL"]:
        grupo = datos_tabla

        if sexo != "TOTAL":
            grupo = grupo.loc[grupo["Sexo"] == sexo]
        if tipo != "TOTAL":
            grupo = grupo.loc[grupo["Tipo_universidad"] == tipo]

        resumen = (grupo.groupby("nivel_academico").agg(Edad_promedio=("edad", "mean"), Estudiantes=("edad", "size"),).reindex(niveles))
        resumen["Estudiantes"] = resumen["Estudiantes"].fillna(0)
        resumen.loc["TOTAL"] = [grupo["edad"].mean(), len(grupo)]
        resultados[(sexo, tipo)] = resumen

tabla_edad_nivel = pd.concat(resultados, axis=1)
tabla_edad_nivel.columns.names = ["Sexo", "Tipo de universidad", "Medida"]
tabla_edad_nivel.index.name = "Nivel académico"

display(tabla_edad_nivel.style.format(
    {
        columna: ("{:.1f}" if columna[2] == "Edad_promedio" else "{:,.0f}")
        for columna in tabla_edad_nivel.columns
    },
    na_rep="—",
))

In [62]:
# Frecuencia de licencia por nivel académico
datos_licencia = pd.concat(
    [
        pregrado_2024_analisis[["nivel_academico", "licencia"]],
        posgrado_2024_analisis[["nivel_academico", "licencia"]],
    ],
    ignore_index=True,
)

tabla_licencia = pd.crosstab(
    datos_licencia["nivel_academico"],
    datos_licencia["licencia"].fillna("Sin información"),
    margins=True,
    margins_name="TOTAL",
)

display(tabla_licencia.style.format("{:,.0f}"))

licencia,Licencia Denegada,Licenciada,TOTAL
nivel_academico,,,
Doctorado,16,"10,025","10,041"
Maestría,81,"77,728","77,809"
Pregrado,"8,275","1,437,093","1,445,368"
Segunda especialidad,17,"21,977","21,994"
TOTAL,"8,389","1,546,823","1,555,212"


In [63]:
# Entidades con licencia denegada y número de registros
datos_entidades = pd.concat(
    [
        pregrado_2024_analisis[["nombre_universidad", "licencia"]],
        posgrado_2024_analisis[["nombre_universidad", "licencia"]],
    ],
    ignore_index=True,
)

tabla_denegadas = (
    datos_entidades.loc[datos_entidades["licencia"].eq("Licencia Denegada"),"nombre_universidad",]
    .value_counts().rename_axis("Nombre de entidad").to_frame("Registros")
)

display(tabla_denegadas.style.format("{:,.0f}"))

,Registros
Nombre de entidad,
Universidad Andina Néstor Cáceres Velásquez,"2,566"
Universidad Alas Peruanas S.A.,"2,525"
Universidad San Pedro,"1,521"
Universidad Científica del Perú,771
Universidad Inca Garcilaso de la Vega Asociación Civil,500
Universidad Privada San Carlos S.A.C.,249
Universidad Latinoamericana CIMA S.A.C.,83
Universidad Particular de Chiclayo,79
Universidad Privada de Pucallpa S.A.C.,62
